# LegalQA Task 2 — High-Performance Dual-T4 GPU Pipeline & Strategy F Ensemble
End-to-end inference using mounted Kaggle Model `qwen-lm/qwen2.5/transformers/3b-instruct` and dataset `phucdangg/legalqa-task2-clean-data`.
- **Architecture**: Exact/Similar QA Memory -> Sparse BM25 + Dense DEk21 v2 -> RRF Fusion -> BGE-Reranker-v2-m3 -> Structured Article Stitcher -> Qwen2.5-3B-Instruct -> Snapping & Strategy F Selection.
- **Hardware**: Dual NVIDIA T4 (GPU 0: Qwen 3B Generator | GPU 1: DEk21 + BGE Reranker | CPU: BM25 + Memory).

In [ ]:
import os, sys, gc, glob, json, zipfile, re, math, time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Ensure repository modules are in Python sys.path
REPO_DIRS = [
    ".",
    "/kaggle/working",
    "/kaggle/working/LegalQA",
    *[d for d in glob.glob("/kaggle/input/**/src", recursive=True)]
]
for r_dir in REPO_DIRS:
    p = os.path.abspath(r_dir if not r_dir.endswith("/src") else os.path.dirname(r_dir))
    if p not in sys.path and os.path.exists(os.path.join(p, "src")):
        sys.path.insert(0, p)
        print(f"Added {p} to sys.path")

# Kaggle Secrets Loading (never hardcode tokens)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

# Device Allocation: GPU 0 for Generator, GPU 1 for Retrieval/Reranking if multi-GPU
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if gpu_count >= 2:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:1"
elif gpu_count == 1:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:0"
else:
    GEN_DEVICE = "cpu"
    RETRIEVAL_DEVICE = "cpu"

print(f"CUDA GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.1f} GB | Compute: sm_{props.major}{props.minor}")
print(f"Assigned -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")

In [ ]:
# Deterministic Artifact & Model Path Discovery
qa_files = glob.glob("/kaggle/input/**/qa_unique.parquet", recursive=True) or glob.glob("artifacts/**/qa_unique.parquet", recursive=True)
chunks_files = glob.glob("/kaggle/input/**/legal_chunks.parquet", recursive=True) or glob.glob("artifacts/**/legal_chunks.parquet", recursive=True)
known_files = glob.glob("/kaggle/input/**/known_qa.json", recursive=True) or glob.glob("artifacts/**/known_qa.json", recursive=True)
test_files = glob.glob("/kaggle/input/**/public-official.json", recursive=True) or glob.glob("artifacts/**/public-official.json", recursive=True)

assert qa_files, "qa_unique.parquet not found!"
assert chunks_files, "legal_chunks.parquet not found!"
assert known_files, "known_qa.json not found!"
assert test_files, "public-official.json not found!"

QA_PATH = qa_files[0]
CHUNKS_PATH = chunks_files[0]
KNOWN_QA_PATH = known_files[0]
TEST_PATH = test_files[0]

# Precomputed Indexes Discovery
bm25_dirs = [d for d in glob.glob("/kaggle/input/**/bm25*", recursive=True) if os.path.isdir(d)] or ["artifacts/task2/indexes/bm25"]
dek21_dirs = [d for d in glob.glob("/kaggle/input/**/dek21*", recursive=True) if os.path.isdir(d)] or ["artifacts/task2/indexes/dek21"]
BM25_DIR = bm25_dirs[0] if bm25_dirs and os.path.exists(bm25_dirs[0]) else "/kaggle/working/indexes/bm25"
DEK21_DIR = dek21_dirs[0] if dek21_dirs and os.path.exists(dek21_dirs[0]) else "/kaggle/working/indexes/dek21"

# Deterministic Qwen Model Discovery
qwen_configs = glob.glob("/kaggle/input/**/qwen*3b*/**/config.json", recursive=True) + \
               glob.glob("/kaggle/input/**/3b-instruct*/**/config.json", recursive=True)
if qwen_configs:
    MODEL_PATH = os.path.dirname(qwen_configs[0])
else:
    MODEL_PATH = "Qwen/Qwen2.5-3B-Instruct"

print(f"Data Root:     {os.path.dirname(QA_PATH)}")
print(f"Chunks Path:   {CHUNKS_PATH}")
print(f"BM25 Dir:      {BM25_DIR}")
print(f"DEk21 Dir:     {DEK21_DIR}")
print(f"Base Model:    {MODEL_PATH}")

In [ ]:
# Import Core Pipeline Modules
from src.common.bm25 import BM25Retriever
from src.common.dense_dek21 import DEk21Retriever
from src.common.reranker import BGEReranker
from src.common.rrf import reciprocal_rank_fusion
from src.task2.article_stitcher import ArticleStitcher
from src.task2.generator import QwenGenerator
from src.task2.qa_memory import QAMemory
from src.task2.source_snap import (
    generate_candidate_ensemble,
    select_best_answer_candidate,
    snap_facts_to_evidence,
)

# 1. Load QA Memory
print("Loading Exact & Similar QA Memory...")
memory = QAMemory.load(KNOWN_QA_PATH, QA_PATH)
print(f"QA Memory loaded with {len(memory.id_to_answer)} IDs and {len(memory.question_to_answer)} questions.")

# 2. Load Sparse BM25
print("Loading Sparse BM25 Retriever...")
if os.path.exists(os.path.join(BM25_DIR, "bm25_manifest.json")):
    bm25 = BM25Retriever.load(BM25_DIR, corpus_path=CHUNKS_PATH)
else:
    print("Precomputed BM25 index not found. Building BM25 index from legal chunks...")
    df_chunks = pd.read_parquet(CHUNKS_PATH, columns=["chunk_id", "text_raw", "text_norm", "doc_name", "parent_article_id", "article_number", "clause_number", "start_char"])
    bm25 = BM25Retriever()
    bm25.fit(df_chunks.to_dict("records"))
    bm25.save(BM25_DIR)
print(f"BM25 Retriever ready ({bm25.corpus_size:,} chunks indexed).")

# 3. Load Dense DEk21 on RETRIEVAL_DEVICE
print(f"Loading Dense DEk21 Retriever on {RETRIEVAL_DEVICE}...")
if os.path.exists(os.path.join(DEK21_DIR, "embeddings.npy")):
    dense = DEk21Retriever.load_index(DEK21_DIR, corpus_path=CHUNKS_PATH, device=RETRIEVAL_DEVICE)
else:
    print("DEk21 precomputed embeddings not found. Initializing encoder...")
    dense = DEk21Retriever(model_name="CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2", device=RETRIEVAL_DEVICE)
    dense.corpus = bm25.corpus
    dense.doc_ids = bm25.doc_ids

# 4. Load Neural Reranker on RETRIEVAL_DEVICE
print(f"Loading BGE-Reranker-v2-m3 on {RETRIEVAL_DEVICE}...")
reranker = BGEReranker(model_name="BAAI/bge-reranker-v2-m3", device=RETRIEVAL_DEVICE)

# 5. Load Article Stitcher
stitcher = ArticleStitcher(bm25.corpus)

# 6. Load Qwen2.5-3B Generator on GEN_DEVICE
print(f"Loading Qwen2.5-3B-Instruct Generator on {GEN_DEVICE}...")
generator = QwenGenerator.load(model_path=MODEL_PATH, device=GEN_DEVICE, runtime="torch")
print("All pipeline modules successfully initialized!")

In [ ]:
# Load Public Test Queries
with open(TEST_PATH, "r", encoding="utf-8") as f:
    public_test = json.load(f)

print(f"Loaded {len(public_test)} public test questions.")
submission = {}
unseen_items = []

# 1. Exact QA Memory Resolution
for qid, item in public_test.items():
    q_text = item.get("question", "")
    exact_ans = memory.lookup_exact(qid, q_text)
    if exact_ans:
        submission[str(qid)] = {"answer": exact_ans}
    else:
        unseen_items.append((str(qid), q_text))

print(f"Exact Memory Hits: {len(submission)} | Unseen Questions to Predict: {len(unseen_items)}")

In [ ]:
# Execute Hybrid Retrieval, Reranking, Stitching & Generation
start_time = time.time()
batch_size = 8 if torch.cuda.is_available() else 1
gen_items = []

print(f"Retrieving and packing evidence for {len(unseen_items)} questions...")
for qid, q in tqdm(unseen_items, desc="Retrieval & Evidence Packing"):
    # 1. Similar QA Memory check
    fuzzy_hit = memory.lookup_fuzzy(q, threshold=0.92)
    fuzzy_ans = fuzzy_hit["answer"] if fuzzy_hit else ""

    # 2. Hybrid Retrieval (BM25 + Dense DEk21)
    bm25_res = bm25.search(q, top_k=50)
    dense_res = dense.search(q, top_k=50) if (dense and dense.corpus_embeddings is not None) else []
    if bm25_res and dense_res:
        fused_res = reciprocal_rank_fusion([bm25_res, dense_res], k=60, weights=[0.5, 0.5])
    else:
        fused_res = bm25_res or dense_res

    # 3. Neural Cross-Encoder Reranking
    top_seeds = reranker.rerank(q, fused_res, top_k=8) if (reranker and fused_res) else fused_res[:8]

    # 4. Structured Article Stitching
    stitched_pkg = stitcher.stitch(top_seeds, max_chars=3500)
    evidence_text = stitched_pkg.get("stitched_text") or (top_seeds[0]["text_raw"] if top_seeds else "")

    doc_name = top_seeds[0].get("doc_name", "") if top_seeds else ""
    art_num = top_seeds[0].get("article_number", "") if top_seeds else ""
    clause_num = top_seeds[0].get("clause_number", "") if top_seeds else ""

    gen_items.append({
        "qid": qid,
        "question": q,
        "evidence": evidence_text,
        "fuzzy_ans": fuzzy_ans,
        "fuzzy_hit": fuzzy_hit,
        "doc_name": doc_name,
        "art_num": art_num,
        "clause_num": clause_num,
    })

print(f"Evidence packing complete. Starting Batched Qwen Generation (B={batch_size})...")

for b_idx in tqdm(range(0, len(gen_items), batch_size), desc="Generating Answers"):
    batch = gen_items[b_idx:b_idx + batch_size]
    pairs = [(it["question"], it["evidence"]) for it in batch]
    gen_answers = generator.generate_batch(pairs, max_new_tokens=384, batch_size=batch_size)

    for it, raw_ans in zip(batch, gen_answers):
        cands = generate_candidate_ensemble(
            gen_ans=raw_ans,
            evidence=it["evidence"],
            exact_ans="",
            fuzzy_ans=it["fuzzy_ans"],
            doc_name=it["doc_name"],
            art_num=it["art_num"],
            clause_num=it["clause_num"],
        )
        selected = select_best_answer_candidate(
            candidates=cands,
            doc_name=it["doc_name"],
            article_num=it["art_num"],
            clause_num=it["clause_num"],
            features=it["fuzzy_hit"],
        )
        submission[it["qid"]] = {"answer": selected}

elapsed = time.time() - start_time
print(f"Inference finished in {elapsed:.1f}s ({len(submission)} total predictions).")

In [ ]:
# Submission Safety Verification & Packaging
assert len(submission) == 1000, f"Expected 1000 items, got {len(submission)}"
test_keys = set(public_test.keys())
sub_keys = set(submission.keys())
assert test_keys == sub_keys, f"Submission keys do not match public test keys! Diff: {test_keys ^ sub_keys}"

for qid, val in submission.items():
    ans = val.get("answer", "")
    assert isinstance(ans, str) and len(ans.strip()) > 0, f"Empty answer for ID {qid}!"
    assert "[DOCUMENT]" not in ans and "[ARTICLE]" not in ans, f"Internal tag found in ID {qid}!"

# Summary Diagnostics
lengths = [len(v["answer"].split()) for v in submission.values()]
print("=== Submission Diagnostics ===")
print(f"Total Queries:      {len(submission):,}")
print(f"Mean Word Count:    {np.mean(lengths):.1f} words")
print(f"Median Word Count:  {np.median(lengths):.1f} words")
print(f"P90 Word Count:     {np.percentile(lengths, 90):.1f} words")
print(f"Min / Max Length:   {np.min(lengths)} / {np.max(lengths)} words")

# Save submission.json and zip
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "artifacts/task2/submissions"
os.makedirs(out_dir, exist_ok=True)
out_json = os.path.join(out_dir, "submission.json")
out_zip = os.path.join(out_dir, "submission.json.zip")

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_json, arcname="submission.json")

print(f"Created final verified submission archive at {out_zip} ({os.path.getsize(out_zip)/1024:.1f} KB)")
print("All submission preflight checks PASSED!")